# 🏢 Главный ноутбук построения модели

Этот ноутбук позволяет:
- Выбрать компанию и режим модели
- Просмотреть канонические формы отчетности
- Просмотреть все настройки из YAML
- Запустить полный pipeline (макро-прогноз → модель → валидация → стресс → рейтинг)
- Просмотреть все чек-листы и результаты

## 📋 Этапы работы:
0. **Просмотр данных**: Канонические формы отчетности (IS/BS/CF)
1. **Конфигурация**: Выбор компании, режима модели, проверка настроек
2. **Макро-прогноз**: Запуск VECM для прогноза макро-факторов
3. **Построение модели**: Стандартный или кастомный режим
4. **Валидация**: Проверка данных, параметров, acceptance checks
5. **Стресс-тест**: Применение стресс-сценариев
6. **Рейтинг**: Расчет базового, прогнозного и стрессового рейтингов


In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY: COMPANY = 'us_steel'  # fallback
print(f'Company: {COMPANY}, ROOT: {ROOT}')


### 🔄 Загрузка данных через Excel-шаблон

1. Сгенерируйте шаблон: `python tools/excel_template_generator.py --output_dir templates/excel_templates`
2. Заполните `template_UNIFIED_All_Data.xlsx` (история, шедулы, сегменты, долг, драйверы)
3. Настройте алиасы в `templates/notebooks/99_Configure_Excel_Loader.ipynb`
4. Загрузите данные для US Steel:
   ```bash
   python tools/load_excel_to_data_mart.py --company us_steel --file data/excel/us_steel_input.xlsx --config configs/excel_loader.yaml --model-type custom --verbose
   ```
5. Макро-факторы (при необходимости):
   ```bash
   python tools/load_macro_excel.py --company us_steel --file data/excel/us_steel_macro.xlsx --scope auto --source manual
   ```

После загрузки перезапустите шаги подготовки истории в этом ноутбуке.


## 🎯 Режимы моделирования: Standard vs Custom

Модель поддерживает два режима работы, настраиваемые через `model.mode` в `project.yaml`:

### Standard Mode (упрощенная методология)
- Revenue: макромодель (VECM/ECM) или CAGR/% роста
- COGS, SG&A: % от Revenue (параметры извлекаются из истории)
- Debt: упрощенная логика (общий долг без ST/LT разделения, auto-draw/sweep)
- PP&E, Taxes, Leases: % методы или отключены
- Gain/Loss on Disposal, Disposal Proceeds: fallback = 0

### Custom Mode (продвинутая методология)
- Все features доступны: PP&E corkscrew, Tax schedule, Lease schedule
- Debt: полное расписание с ST/LT разделением, covenants, рефинансированием
- Gain/Loss on Disposal, Disposal Proceeds: из PP&E schedule

**Настройка:** `model.mode` в `project.yaml` (standard | custom)

**Документация:** 
- `docs/MODEL_ENGINES_COMPARISON.md` - детальное сравнение режимов
- `docs/ENGINE_CONFIGURATION_MATRIX_ANALYSIS.md` - соответствие Engine Configuration Matrix


In [ ]:
# Импорты и настройка
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import json
from typing import Optional
from IPython.display import display, Markdown, HTML

from engine.database.data_mart import get_data_mart
from engine.orchestrator import run_all

# Примечание о новой архитектуре:
# Витрина данных (FinancialDataMart) использует модульную архитектуру:
# - Внутри используются repositories (history_repository, canonical_repository, macro_repository и т.д.)
# - Внутри используются services (normalization_service, validation_service)
# - Внешний API остался прежним для обратной совместимости
# - Можно использовать repositories напрямую: mart.macro_repository, mart.history_repository и т.д.
from engine.acceptance.checks import comprehensive_data_validation_and_parameters
from engine.stress_rating.core import run_stress, run_rating

ROOT = Path('../..').resolve()
print(f"ROOT: {ROOT}")
print(f"Companies available: {[d.name for d in (ROOT / 'companies').iterdir() if d.is_dir()]}")

def _canonical_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    if {'metric', 'year', 'value'}.issubset(df.columns):
        return df.pivot(index='metric', columns='year', values='value').sort_index()
    if 'metric' in df.columns:
        return df.set_index('metric')
    return df

def _format_table(df: pd.DataFrame) -> pd.DataFrame:
    formatted = df.copy()
    for col in formatted.columns:
        if pd.api.types.is_numeric_dtype(formatted[col]):
            formatted[col] = formatted[col].apply(
                lambda x: f"{x:,.0f}" if pd.notna(x) and abs(x) >= 1 else (f"{x:,.2f}" if pd.notna(x) else "")
            )
    return formatted


def _get_latest_version(mart) -> Optional[str]:
    versions = mart.get_existing_versions()
    if not versions:
        return None
    return versions[0].get('version')

# ===== ПРОСМОТР КАНОНИЧЕСКИХ ФОРМ ОТЧЕТНОСТИ =====
display(Markdown("---\n## 📊 Канонические формы отчетности"))

COMPANY = COMPANY  # Можно изменить
mart = get_data_mart(ROOT, COMPANY)
try:
    hist_is = _canonical_table(mart.get_history_income_statement(canonical=True))
    hist_bs = _canonical_table(mart.get_history_balance_sheet(canonical=True))
    hist_cf = _canonical_table(mart.get_history_cash_flow(canonical=True))

    if not hist_is.empty:
        display(Markdown(f"### 📈 Income Statement (IS) - {COMPANY.upper()}"))
        display(_format_table(hist_is))
    else:
        display(Markdown("❌ История IS не найдена в витрине"))

    if not hist_bs.empty:
        display(Markdown(f"### 💼 Balance Sheet (BS) - {COMPANY.upper()}"))
        display(_format_table(hist_bs))
    else:
        display(Markdown("❌ История BS не найдена в витрине"))

    if not hist_cf.empty:
        display(Markdown(f"### 💰 Cash Flow Statement (CF) - {COMPANY.upper()}"))
        display(_format_table(hist_cf))
    else:
        display(Markdown("❌ История CF не найдена в витрине"))
finally:
    mart.close()


## 0️⃣ Создание новой модели (или перейдите к разделу 1️⃣)

Если нужно создать модель для новой компании, начните с этого раздела.


In [ ]:
# ===== СОЗДАНИЕ НОВОЙ МОДЕЛИ =====
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout

# Виджеты для создания новой компании
new_company_name = widgets.Text(
    value='',
    placeholder='new_company',
    description='Название компании:',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_industry = widgets.Dropdown(
    options=['metallurgy', 'aluminum', 'steel', 'oil_gas', 'mining', 'retail', 'other'],
    value='metallurgy',
    description='Отрасль:',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_currency = widgets.Dropdown(
    options=['USD', 'RUB', 'EUR', 'CNY'],
    value='USD',
    description='Валюта:',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_mode = widgets.Dropdown(
    options=['standard', 'custom'],
    value='standard',
    description='Режим модели:',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_macro_factors = widgets.Textarea(
    value='gdp_us, industrial_production_us, cpi_us, ppi_us',
    placeholder='Список факторов через запятую',
    description='Макро-факторы:',
    layout=Layout(width='600px', height='80px'),
    style={'description_width': '200px'}
)

new_company_revenue_driver = widgets.Text(
    value='',
    placeholder='steel_price_hrc или gdp_us',
    description='Основной драйвер выручки:',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_rc_limit = widgets.FloatText(
    value=1000.0,
    description='RC Limit (млн):',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

new_company_min_cash = widgets.FloatText(
    value=100.0,
    description='Min Cash (млн):',
    style={'description_width': '200px'},
    layout=Layout(width='400px')
)

create_button = widgets.Button(
    description='🚀 Создать новую модель',
    button_style='success',
    icon='plus'
)

create_output = widgets.Output()

def create_new_company_model(b):
    """Создание новой модели компании"""
    global COMPANY, MODEL_MODE, croot, proj_yaml
    
    company_name = new_company_name.value.strip()
    if not company_name:
        with create_output:
            create_output.clear_output()
            print("❌ Ошибка: Укажите название компании")
        return
    
    # Проверка что компания не существует
    existing_company_dir = ROOT / "companies" / company_name
    if existing_company_dir.exists():
        with create_output:
            create_output.clear_output()
            print(f"⚠️ Компания '{company_name}' уже существует. Выберите другое название.")
        return
    
    industry = new_company_industry.value
    currency = new_company_currency.value
    mode = new_company_mode.value
    factors_list = [f.strip() for f in new_company_macro_factors.value.split(',') if f.strip()]
    
    with create_output:
        create_output.clear_output()
        print(f"📦 Создание модели для компании: {company_name}")
        print(f"   Отрасль: {industry}")
        print(f"   Валюта: {currency}")
        print(f"   Режим: {mode}")
        print(f"   Макро-факторы: {len(factors_list)}")
        
        # Создание структуры директорий
        dirs_to_create = [
            ROOT / "companies" / company_name / "configs",
            ROOT / "companies" / company_name / "history",
            ROOT / "companies" / company_name / "data" / "debt",
            ROOT / "companies" / company_name / "data" / "operational",
            ROOT / "companies" / company_name / "data" / "macro",
            ROOT / "companies" / company_name / "drivers",
            ROOT / "companies" / company_name / "notebooks",
            ROOT / "companies" / company_name / "outputs" / "checks",
            ROOT / "companies" / company_name / "outputs" / "logs",
            ROOT / "companies" / company_name / "outputs" / "model",
            ROOT / "companies" / company_name / "outputs" / "macro_forecast",
            ROOT / "companies" / company_name / "outputs" / "stress",
            ROOT / "companies" / company_name / "outputs" / "rating",
        ]
        
        for d in dirs_to_create:
            d.mkdir(parents=True, exist_ok=True)
            print(f"   ✅ Создана директория: {d.relative_to(ROOT)}")
        
        # Загрузка шаблона project.yaml
        template_path = ROOT / "templates" / "project_template.yaml"
        if template_path.exists():
            with open(template_path, 'r', encoding='utf-8') as f:
                template_content = f.read()
            
            # Замена плейсхолдеров
            template_content = template_content.replace('{company}', company_name)
            template_content = template_content.replace('{industry}', industry)
            
            # Создание project.yaml
            proj_yaml_path = ROOT / "companies" / company_name / "configs" / "project.yaml"
            with open(proj_yaml_path, 'w', encoding='utf-8') as f:
                f.write(template_content)
            
            # Загрузка и кастомизация YAML
            config = yaml.safe_load(template_content)
            
            # Обновление макро-факторов
            config['macro_forecast']['factors'] = factors_list
            
            # Обновление file_map (упрощенная версия - пользователь может настроить позже)
            file_map = {}
            for factor in factors_list:
                file_map[factor] = f"{factor}.csv"
            config['macro_forecast']['file_map'] = file_map
            
            # Обновление search_paths
            config['macro_forecast']['search_paths'] = [
                f"companies/{company_name}/drivers",
                f"companies/{company_name}/data/macro",
                f"macro/industry/{industry}/drivers",
                "macro/global/drivers"
            ]
            
            # Обновление history paths
            config['history'] = {
                'is': f"companies/{company_name}/history/is_history_{company_name}.csv",
                'bs': f"companies/{company_name}/history/bs_history_{company_name}.csv",
                'cf': f"companies/{company_name}/history/cf_history_{company_name}.csv"
            }
            
            # Режим модели
            config['model']['mode'] = mode
            
            # Дополнительные настройки если указаны
            revenue_driver = new_company_revenue_driver.value.strip()
            if revenue_driver and revenue_driver in factors_list:
                if 'revenue' not in config['model']['standard']:
                    config['model']['standard']['revenue'] = {}
                config['model']['standard']['revenue']['driver'] = revenue_driver
            
            # Настройки RC
            if 'debt' not in config['model']['standard']:
                config['model']['standard']['debt'] = {}
            if 'rc' not in config['model']['standard']['debt']:
                config['model']['standard']['debt']['rc'] = {}
            
            config['model']['standard']['debt']['rc']['limit'] = new_company_rc_limit.value
            config['model']['standard']['debt']['rc']['min_cash'] = new_company_min_cash.value
            
            # Features
            if 'features' not in config:
                config['features'] = {}
            config['features']['min_cash'] = new_company_min_cash.value
            
            # Сохранение обновленного конфига
            with open(proj_yaml_path, 'w', encoding='utf-8') as f:
                yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2)
            
            print(f"   ✅ Создан project.yaml: {proj_yaml_path}")
            
            # Создание README
            readme_path = ROOT / "companies" / company_name / "README.md"
            readme_content = f"""# {company_name.upper()} - Модель финансового прогнозирования

## 📋 Общая информация

- **Компания**: {company_name}
- **Отрасль**: {industry}
- **Валюта**: {currency}
- **Режим модели**: {mode}

## 📁 Структура

```
companies/{company_name}/
├── configs/
│   └── project.yaml          # Главный конфигурационный файл
├── history/                  # Историческая финансовая отчетность
│   ├── is_history_{company_name}.csv
│   ├── bs_history_{company_name}.csv
│   └── cf_history_{company_name}.csv
├── data/
│   ├── debt/                 # Структура долга
│   ├── operational/          # Операционные показатели
│   └── macro/               # Локальные макро-факторы (если есть)
├── drivers/                 # Компания-специфичные драйверы
├── notebooks/               # Jupyter ноутбуки
└── outputs/                 # Результаты расчетов
```

## 🚀 Быстрый старт

1. **Подготовьте исторические данные**:
   - Разместите IS/BS/CF в папке `history/`
   - Формат: CSV с колонкой `year` или широкий формат с годами как колонки

2. **Настройте макро-факторы**:
   - Разместите файлы макро-факторов в соответствующих папках из `search_paths`
   - Обновите `file_map` в `project.yaml` если нужно

3. **Запустите модель**:
   ```python
   from pathlib import Path
   from engine.orchestrator import run_all
   
   root = Path('.')
   run_all(root, '{company_name}', run_id='INITIAL_RUN')
   ```

## ⚙️ Настройки

Основные настройки находятся в `configs/project.yaml`:
- Макро-факторы и их файлы
- Параметры моделирования Revenue
- Настройки Debt & Refinancing
- Working Capital параметры
- Train/Test Split для валидации

Для интерактивной настройки используйте ноутбук `notebooks/99_Configure_YAML.ipynb`.

## 📊 Макро-факторы

Текущие факторы: {', '.join(factors_list)}

Файлы должны быть размещены в одной из папок из `search_paths`:
{chr(10).join(['- ' + sp for sp in config['macro_forecast']['search_paths']])}

## ✅ Следующие шаги

1. **Заполните Excel шаблоны**:
   - Откройте `templates/` для Excel шаблонов
   - Заполните данные в Excel (IS/BS/CF, макро-факторы)
   - Используйте `python tools/excel_loader.py` для конвертации в CSV
   - Пример: `python tools/excel_loader.py --input templates/ --output ../ --company {company_name}`

2. **Настройте конфигурацию**:
   - Откройте `notebooks/99_Configure_YAML.ipynb`
   - Или редактируйте `configs/project.yaml` вручную

3. **Запустите валидацию и модель**:
   - Откройте `notebooks/00_Build_Model_Main.ipynb`
   - Запустите валидацию данных
   - Запустите полный pipeline

Подробные инструкции: см. `templates/excel_templates/README.md`
"""
            with open(readme_path, 'w', encoding='utf-8') as f:
                f.write(readme_content)
            print(f"   ✅ Создан README.md: {readme_path}")
            
            # Создание шаблонных CSV файлов истории (с заголовками)
            history_templates = {
                'is': ['year,revenue,cogs,sga,depreciation,interest_expense,tax_expense,net_income'],
                'bs': ['year,cash,receivables,inventory,payables,ppe,debt,equity'],
                'cf': ['year,cfo,cfi,cff,net_change_in_cash']
            }
            
            for stmt_type, header in history_templates.items():
                hist_file = ROOT / "companies" / company_name / "history" / f"{stmt_type}_history_{company_name}.csv"
                with open(hist_file, 'w', encoding='utf-8') as f:
                    f.write(header[0] + '\n')
                    # Добавляем пример строки для одного года
                    f.write(f"2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n")
                print(f"   ✅ Создан шаблон {stmt_type}_history_{company_name}.csv")
            
            # Копирование ноутбуков
            notebooks_to_copy = [
                ("00_Build_Model_Main.ipynb", True),
                ("01_Test_Macro_Module.ipynb", True),  # Тестирование макропрогноза
                ("02_Test_Model_Module.ipynb", True),  # Тестирование модели
                ("99_Configure_YAML.ipynb", False),  # опционально, если существует
            ]
            
            for nb_name, required in notebooks_to_copy:
                try:
                    # Пробуем найти ноутбук в нескольких местах (приоритет: templates, потом примеры)
                    sources = [
                        ROOT / "templates" / "notebooks" / nb_name,  # ПЕРВЫЙ ПРИОРИТЕТ: шаблоны
                        ROOT / "companies" / "us_steel" / "notebooks" / nb_name,
                        ROOT / "companies" / "rusal" / "notebooks" / nb_name,
                    ]
                    
                    notebook_src = None
                    for src in sources:
                        if src.exists():
                            notebook_src = src
                            break
                    
                    if notebook_src:
                        notebook_dst = ROOT / "companies" / company_name / "notebooks" / nb_name
                        import shutil
                        shutil.copy(notebook_src, notebook_dst)
                        print(f"   ✅ Скопирован ноутбук: {nb_name}")
                    elif required:
                        print(f"   ⚠️ Ноутбук {nb_name} не найден (требуется)")
                except Exception as e:
                    if required:
                        print(f"   ⚠️ Не удалось скопировать ноутбук {nb_name}: {e}")
            
            
            # Копирование Excel шаблонов для загрузки данных
            excel_templates_dir = ROOT / "templates" / "excel_templates"
            company_excel_dir = ROOT / "companies" / company_name / "templates"
            company_excel_dir.mkdir(parents=True, exist_ok=True)
            
            excel_templates_to_copy = [
                "template_UNIFIED_All_Data.xlsx",
                "template_IS_Income_Statement.xlsx",
                "template_BS_Balance_Sheet.xlsx",
                "template_CF_Cash_Flow.xlsx",
                "template_MACRO_Factor.xlsx",
                "template_DEBT_Schedule.xlsx",
                "template_OPERATIONAL_Metric.xlsx",
                "README.md"
            ]
            
            copied_excel = 0
            for tmpl_name in excel_templates_to_copy:
                tmpl_src = excel_templates_dir / tmpl_name
                if tmpl_src.exists():
                    tmpl_dst = company_excel_dir / tmpl_name
                    import shutil
                    shutil.copy(tmpl_src, tmpl_dst)
                    copied_excel += 1
            
            if copied_excel > 0:
                print(f"   ✅ Скопировано Excel шаблонов: {copied_excel}")
            
            # Создание пустого debt_schedule.csv с заголовками
            debt_schedule_path = ROOT / "companies" / company_name / "data" / "debt" / "debt_schedule.csv"
            debt_header = "instrument,instrument_type,rate,rate_type,bal,start,end,payment_type,limit,min_cash,rate_spread\n"
            with open(debt_schedule_path, 'w', encoding='utf-8') as f:
                f.write(debt_header)
                # Пример RC линии с параметрами из виджетов
                f.write(f"RC_Facility_{company_name},RC,0.05,fixed,0,2010,2030,bullet,{new_company_rc_limit.value},{new_company_min_cash.value},0.03\n")
            print(f"   ✅ Создан шаблон debt_schedule.csv")
            
            # Регистрация компании в единой централизованной БД
            try:
                from engine.database import get_central_db
                db = get_central_db(ROOT)
                db.register_company(
                    company_name=company_name,
                    industry=industry,
                    currency=currency,
                    model_mode=mode
                )
                print(f"   ✅ Компания зарегистрирована в централизованной БД")
                
                # Создание витрины данных для компании
                db.create_company_data_mart_views(company_name)
                print(f"   ✅ Создана витрина данных для компании")
                
                db.close()
            except Exception as e:
                print(f"   ⚠️ Предупреждение: Не удалось зарегистрировать компанию в БД: {e}")
            
            # Обновление глобальных переменных для автоматического переключения
            global COMPANY
            COMPANY = company_name
            print(f"\n💡 Совет: Обновите переменную COMPANY в следующей ячейке на '{company_name}' для продолжения работы")
            
            print(f"\n✅ Модель для компании '{company_name}' успешно создана!")
            print(f"\n✅ Модель для компании '{company_name}' успешно создана!")
            print(f"\n📝 Следующие шаги:")
            print(f"   1. 📊 Заполните Excel шаблоны:")
            print(f"      • Откройте templates/excel_templates/ для Excel шаблонов")
            print(f"      • Используйте python tools/excel_loader.py для конвертации")
            print(f"   2. 📁 Добавьте исторические данные:")
            print(f"      • IS/BS/CF в history/")
            print(f"      • Макро-факторы в drivers/ или macro/")
            print(f"   3. ⚙️  Настройте конфигурацию:")
            print(f"      • Откройте notebooks/99_Configure_YAML.ipynb")
            print(f"      • Или редактируйте configs/project.yaml вручную")
            print(f"   4. 🔍 Запустите валидацию данных в этом ноутбуке")
            print(f"   5. 🚀 Запустите модель")
        else:
            print(f"❌ Шаблон не найден: {template_path}")

create_button.on_click(create_new_company_model)

# Отображение виджетов создания
create_section = VBox([
    Markdown("### 🆕 Создание новой модели компании"),
    widgets.HTML("<b>Основная информация:</b>"),
    new_company_name,
    new_company_industry,
    new_company_currency,
    new_company_mode,
    widgets.HTML("<br><b>Макро-факторы:</b>"),
    new_company_macro_factors,
    widgets.HTML("<i>Укажите макро-факторы через запятую (например: gdp_us, cpi_us, steel_price_hrc)</i>"),
    widgets.HTML("<br><b>Параметры моделирования (опционально):</b>"),
    new_company_revenue_driver,
    widgets.HTML("<i>Основной драйвер выручки (должен быть в списке макро-факторов)</i>"),
    new_company_rc_limit,
    new_company_min_cash,
    widgets.HTML("<br>"),
    create_button,
    create_output
])

display(create_section)

# ===== КОНФИГУРАЦИЯ (для существующих компаний) =====
display(Markdown("---\n## 1️⃣ Конфигурация: Выбор компании и режима"))

# Измените эти параметры:
COMPANY = COMPANY  # или "rusal" или имя новой компании
MODEL_MODE = "standard"  # "standard" или "custom"
RUN_ID = "NOTEBOOK_RUN_01"

# Пути
croot = ROOT / "companies" / COMPANY
proj_yaml_path = croot / "configs" / "project.yaml"

# Загрузка конфигурации
if proj_yaml_path.exists():
    with open(proj_yaml_path, 'r', encoding='utf-8') as f:
        proj_yaml = yaml.safe_load(f) or {}
    
    # Обновление режима модели если нужно
    if MODEL_MODE:
        if "model" not in proj_yaml:
            proj_yaml["model"] = {}
        proj_yaml["model"]["mode"] = MODEL_MODE
        # Сохраняем обновленную конфигурацию
        with open(proj_yaml_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(proj_yaml, f, allow_unicode=True, sort_keys=False)
    
    print(f"✅ Компания: {COMPANY}")
    print(f"✅ Режим модели: {proj_yaml.get('model', {}).get('mode', 'standard')}")
    print(f"✅ Конфигурация загружена из: {proj_yaml_path}")
else:
    print(f"❌ Конфигурация не найдена: {proj_yaml_path}")
    print(f"💡 Используйте раздел 'Создание новой модели' выше для создания новой компании")
    proj_yaml = {}


## 2️⃣ Просмотр настроек из YAML

### Основные параметры моделирования


In [ ]:
# Вывод ключевых настроек
def print_yaml_section(yaml_dict, section_path, title):
    """Рекурсивно выводит секцию YAML"""
    keys = section_path.split('.')
    current = yaml_dict
    for key in keys:
        current = current.get(key, {})
    
    if current:
        display(Markdown(f"### {title}"))
        print(yaml.dump(current, allow_unicode=True, sort_keys=False, default_flow_style=False))
        print("\n")

# Вывод ключевых разделов
if proj_yaml:
    print_yaml_section(proj_yaml, "model.standard.revenue", "📊 Revenue Modeling")
    print_yaml_section(proj_yaml, "model.standard.debt", "💳 Debt & Refinancing")
    print_yaml_section(proj_yaml, "model.standard.train_test", "📈 Train/Test Split")
    print_yaml_section(proj_yaml, "features", "⚙️ Feature Flags")


In [ ]:
# Хелперы для чтения legacy CSV

def load_company_csv(relative_path: str) -> pd.DataFrame:
    """Загружает CSV из директории компании, если витрина пуста."""
    csv_path = croot / relative_path
    if csv_path.exists():
        print(f"⚠️ Используем legacy CSV: {csv_path.relative_to(ROOT)}")
        try:
            return pd.read_csv(csv_path)
        except Exception as exc:
            print(f"⚠️ Ошибка чтения {relative_path}: {exc}")
            return pd.DataFrame()
    print(f"ℹ️ Legacy CSV отсутствует: {relative_path}")
    return pd.DataFrame()



## 3️⃣ Валидация данных и параметры

Проверка исторических данных и расчет параметров на основе истории.


In [ ]:
# Комплексная валидация данных
print("🔍 Запуск валидации данных...")
validation_success = comprehensive_data_validation_and_parameters(croot)

if validation_success:
    print("✅ Валидация завершена")

    with get_data_mart(ROOT, COMPANY) as mart:
        version = MODEL_VERSION or _get_latest_version(mart)
        if version:
            validation_df = pd.read_sql_query(
                "SELECT metric, year, value FROM model_results WHERE company = ? AND version = ? AND statement_type = ?",
                mart.db.conn,
                params=(COMPANY, version, "DATA_VALIDATION"),
            )
            if not validation_df.empty:
                display(Markdown("### Результаты валидации данных"))
                display(validation_df)

            params_df = pd.read_sql_query(
                "SELECT parameter_name, parameter_value FROM model_parameters WHERE company = ? AND version = ? AND parameter_type = ?",
                mart.db.conn,
                params=(COMPANY, version, "history_parameter"),
            )
            if not params_df.empty:
                display(Markdown("### Параметры моделирования (из истории)"))
                display(params_df)

    # Legacy fallback, если витрина не заполнена
    validation_df = load_company_csv("outputs/checks/data_validation.csv")
    if not validation_df.empty:
        display(Markdown("### ♻️ Результаты валидации (legacy csv)"))
        display(validation_df)

    params_df = load_company_csv("outputs/checks/parameters_summary.csv")
    if not params_df.empty:
        display(Markdown("### ♻️ Параметры моделирования (legacy csv)"))
        display(params_df)
else:
    print("⚠️ Валидация завершена с предупреждениями")


## 3.5️⃣ Комплексное тестирование данных и отслеживание мэппинга метрик

Перед запуском модели рекомендуется провести комплексное тестирование данных и отслеживание мэппинга метрик от БД через канон в движок.


In [ ]:
# Комплексное тестирование данных и отслеживание мэппинга метрик
from tools.test_suite import TestSuite
from tools.metric_mapping_tracker import MetricMappingTracker
from engine.notebook_helpers import display_test_results
from IPython.display import display, Markdown
from engine.model3.io import load_inputs
from engine.model3.core import ThreeStatementModel

# Настройки
TEST_YEARS = list(range(2010, 2025))
RUN_TESTS = True  # Запустить тестирование
TRACK_METRIC_MAPPING = True  # Отслеживать мэппинг метрик

if RUN_TESTS:
    print("🧪 Комплексное тестирование данных...")
    print(f"📊 Компания: {COMPANY}")
    print(f"📅 Годы: {TEST_YEARS[0]}-{TEST_YEARS[-1]}")
    
    # Создаем TestSuite
    suite = TestSuite(company=COMPANY, root=ROOT, years=TEST_YEARS)
    result = suite.run_all_tests()
    
    # Отображаем результаты
    display_test_results({
        'total_tests': result.total_tests,
        'passed': result.passed,
        'failed': result.failed,
        'warnings': result.warnings,
        'skipped': result.skipped,
        'results': [
            {
                'name': r.name,
                'status': r.status,
                'message': r.message,
                'details': r.details
            }
            for r in result.results
        ]
    })
    
    # Сохраняем отчет
    report_path = ROOT / "companies" / COMPANY / "outputs" / "test_report.json"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    suite.generate_report(report_path)
    print(f"\n📄 Отчет сохранен: {report_path}")

# Отслеживание мэппинга метрик: БД → Канон → Движок
if TRACK_METRIC_MAPPING:
    print("\n" + "="*80)
    print("🔍 ОТСЛЕЖИВАНИЕ МЭППИНГА МЕТРИК: БД → КАНОН → ДВИЖОК")
    print("="*80)
    
    try:
        base_year = max(TEST_YEARS) if TEST_YEARS else 2024
        
        # Инициализация трекера
        tracker = MetricMappingTracker(company=COMPANY, root=ROOT)
        tracker.load_mappings()
        print(f"✓ Загружено {len(tracker.mappings)} метрик в мэппинге")
        
        # Отслеживание метрик из БД
        tracker.track_db_metrics(base_year)
        db_count = sum(1 for m in tracker.mappings.values() if m.found_in_db)
        print(f"✓ Найдено {db_count} метрик в БД")
        
        # Загрузка данных для отслеживания
        print("\n📥 Загрузка данных для отслеживания...")
        mart = get_data_mart(ROOT, COMPANY)
        hist_state, history, drivers, canonical = load_inputs(
            root=ROOT,
            company=COMPANY,
            use_canonical=True,
            use_new_schema=True
        )
        
        # Отслеживание метрик из Canonical
        tracker.track_canonical_metrics(canonical, base_year)
        canonical_count = sum(1 for m in tracker.mappings.values() if m.found_in_canonical)
        print(f"✓ Найдено {canonical_count} метрик в Canonical")
        
        # Отслеживание метрик из HistoricState
        tracker.track_historic_state_metrics(hist_state, base_year)
        hist_count = sum(1 for m in tracker.mappings.values() if m.found_in_historic_state)
        print(f"✓ Найдено {hist_count} метрик в HistoricState")
        
        # Построение модели для отслеживания метрик в движке
        print("\n🔧 Построение модели для отслеживания метрик в движке...")
        model = ThreeStatementModel(
            hist=hist_state,
            forecast_years=[base_year + 1, base_year + 2, base_year + 3],
            drv=drivers,
            canonical=canonical,
            mart=mart
        )
        model.run()
        
        # Отслеживание метрик из движка
        tracker.track_engine_metrics(model, base_year)
        engine_count = sum(1 for m in tracker.mappings.values() if m.found_in_engine)
        print(f"✓ Найдено {engine_count} метрик в Движке")
        
        # Анализ потока данных
        print("\n📊 Анализ потока данных...")
        analysis = tracker.analyze_mapping_flow(base_year)
        
        # Отображение результатов
        display(Markdown("### 📊 Статистика мэппинга метрик"))
        stats_df = pd.DataFrame({
            'Этап': ['БД', 'Canonical', 'HistoricState', 'Движок'],
            'Количество метрик': [
                analysis['metrics_by_stage']['db'],
                analysis['metrics_by_stage']['canonical'],
                analysis['metrics_by_stage']['historic_state'],
                analysis['metrics_by_stage']['engine']
            ]
        })
        display(stats_df)
        
        # Потерянные метрики
        if analysis['lost_metrics']:
            display(Markdown(f"### ⚠️ Потерянные метрики: {len(analysis['lost_metrics'])}"))
            lost_df = pd.DataFrame(analysis['lost_metrics'][:20])
            display(lost_df[['metric', 'statement', 'lost_at']])
            if len(analysis['lost_metrics']) > 20:
                print(f"... и еще {len(analysis['lost_metrics']) - 20} метрик")
        
        # Расхождения
        if analysis['discrepancies']:
            display(Markdown(f"### 🔍 Расхождения значений: {len(analysis['discrepancies'])}"))
            disc_df = pd.DataFrame(analysis['discrepancies'][:20])
            disc_df['difference'] = disc_df['difference'].apply(lambda x: f"{x:,.0f}")
            disc_df['relative_diff_pct'] = disc_df['relative_diff_pct'].apply(lambda x: f"{x:.2f}%")
            display(disc_df[['metric', 'statement', 'db_value', 'engine_value', 'difference', 'relative_diff_pct']])
            if len(analysis['discrepancies']) > 20:
                print(f"... и еще {len(analysis['discrepancies']) - 20} расхождений")
        
        # Combine метрики
        if analysis['combine_metrics']:
            display(Markdown(f"### 🔗 Combine метрики: {len(analysis['combine_metrics'])}"))
            combine_df = pd.DataFrame(analysis['combine_metrics'][:20])
            display(combine_df[['metric', 'statement', 'combine_from']])
            if len(analysis['combine_metrics']) > 20:
                print(f"... и еще {len(analysis['combine_metrics']) - 20} combine метрик")
        
        # Генерация полного отчета
        print("\n📄 Генерация полного отчета...")
        report = tracker.generate_mapping_report(base_year)
        
        # Сохранение отчета
        report_path = ROOT / "companies" / COMPANY / "outputs" / f"metric_mapping_report_{base_year}.txt"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(report, encoding='utf-8')
        print(f"✅ Отчет сохранен: {report_path.relative_to(ROOT)}")
        
        # Отображение краткого отчета
        display(Markdown("### 📋 Краткий отчет о мэппинге"))
        print(report[:2000])
        print(f"\n... (полный отчет сохранен в {report_path.relative_to(ROOT)})")
        
        mart.close()
        
    except Exception as e:
        print(f"⚠️ Ошибка при отслеживании мэппинга метрик: {e}")
        import traceback
        traceback.print_exc()


## 4️⃣ Запуск полного pipeline

Запускает: макро-прогноз → модель → валидация → стресс → рейтинг


In [ ]:
# Получение последней версии модели (при наличии)
MODEL_VERSION = None

with get_data_mart(ROOT, COMPANY) as mart:
    MODEL_VERSION = _get_latest_version(mart)

print(f"MODEL_VERSION (latest): {MODEL_VERSION}")


In [ ]:
# Запуск полного pipeline
print(f"🚀 Запуск полного pipeline для компании: {COMPANY}")
print(f"   Режим: {MODEL_MODE}")
print(f"   Run ID: {RUN_ID}")
print()

success = False
try:
    success = run_all(ROOT, COMPANY, run_id=RUN_ID)
    if success:
        print(f"\n✅ Pipeline успешно завершен!")
    else:
        print(f"\n⚠️ Pipeline завершен с предупреждениями")
except Exception as e:
    print(f"\n❌ Ошибка при запуске pipeline: {e}")
    import traceback
    traceback.print_exc()
finally:
    with get_data_mart(ROOT, COMPANY) as mart:
        MODEL_VERSION = _get_latest_version(mart)
    print(f"\n📦 Текущая версия модели: {MODEL_VERSION}")


## 5️⃣ Acceptance Checks (Чек-листы)

Проверка балансировки, cash bridge, валидация прогнозов


In [ ]:
# Показать ключевые проверки из витрины данных

def _load_model_checks(mart, version: str) -> pd.DataFrame:
    query = (
        "SELECT parameter_name, parameter_value "
        "FROM model_parameters "
        "WHERE company = ? AND version = ? AND parameter_type = ? "
        "ORDER BY parameter_name"
    )
    return pd.read_sql_query(query, mart.db.conn, params=(COMPANY, version, "model_check"))

with get_data_mart(ROOT, COMPANY) as mart:
    version = MODEL_VERSION or _get_latest_version(mart)
    if not version:
        print("⚠️ Версия модели не найдена. Сначала запустите pipeline.")
    else:
        display(Markdown(f"### ✅ Проверки модели (версия: {version})"))

        checks_df = _load_model_checks(mart, version)
        if not checks_df.empty:
            display(Markdown("#### Model Checks"))
            display(checks_df)
        else:
            print("⚠️ Model checks не найдены в витрине (версия не сохраняла проверки).")

        covenants_df = mart.get_model_results(version, "COVENANTS", canonical=False)
        if not covenants_df.empty:
            display(Markdown("#### Covenants"))
            display(covenants_df)

        stress_df = mart.get_model_results(version, "STRESS", canonical=False)
        if not stress_df.empty:
            display(Markdown("#### Stress Schedules"))
            display(stress_df)

        rating_df = mart.get_model_results(version, "RATING", canonical=False)
        if not rating_df.empty:
            display(Markdown("#### Rating Summary"))
            display(rating_df)

# Legacy fallback (при необходимости)
legacy_files = {
    "outputs/checks/acceptance_checks.csv": "Acceptance Checks (legacy)",
    "outputs/checks/forecast_validation_metrics.csv": "Forecast Validation (legacy)",
}
for relative_path, title in legacy_files.items():
    df = load_company_csv(relative_path)
    if not df.empty:
        display(Markdown(f"### ♻️ {title}"))
        display(df)
        print("ℹ️ Используется legacy CSV. Рекомендуется перейти на витрину данных.")


## 6️⃣ Результаты модели: IS/BS/CF

Просмотр прогноза финансовых отчетов


In [ ]:
# Показать результаты модели из витрины
with get_data_mart(ROOT, COMPANY) as mart:
    version = MODEL_VERSION or _get_latest_version(mart)
    if not version:
        print("⚠️ Версия модели не найдена. Запустите pipeline.")
    else:
        for statement in ["IS", "BS", "CF", "DEBT"]:
            df = mart.get_model_results(version, statement, canonical=(statement in {"IS", "BS", "CF"}))
            if df.empty:
                continue
            display(Markdown(f"### 📄 {statement} Statement"))
            display(df)

# Legacy fallback: показываем CSV только если нужно
legacy_files = {
    "outputs/model/3statement_IS.csv": "IS Statement (legacy)",
    "outputs/model/3statement_BS.csv": "BS Statement (legacy)",
    "outputs/model/3statement_CF.csv": "CF Statement (legacy)",
}
for relative_path, title in legacy_files.items():
    df = load_company_csv(relative_path)
    if not df.empty:
        display(Markdown(f"### ♻️ {title}"))
        display(df)
        print("ℹ️ Используется legacy CSV. Рекомендуется использовать витрину.")


## 7️⃣ Стресс-тест и Рейтинг

Расчет базового, прогнозного и стрессового рейтингов


In [ ]:
# Запуск стресс-теста и рейтинга (через витрину)
print("💥 Запуск стресс-теста...")
try:
    run_stress(ROOT, COMPANY, version=MODEL_VERSION)
    print("✅ Стресс-тест завершен")
except Exception as e:
    print(f"⚠️ Ошибка стресс-теста: {e}")

print("\n⭐ Запуск расчета рейтинга...")
try:
    run_rating(ROOT, COMPANY, version=MODEL_VERSION)
    print("✅ Рейтинг рассчитан")
except Exception as e:
    print(f"⚠️ Ошибка расчета рейтинга: {e}")

with get_data_mart(ROOT, COMPANY) as mart:
    version = MODEL_VERSION or _get_latest_version(mart)
    if version:
        stress_df = mart.get_model_results(version, "STRESS", canonical=False)
        if not stress_df.empty:
            display(Markdown("### 💥 Стресс-сценарии"))
            display(stress_df)

        rating_df = mart.get_model_results(version, "RATING", canonical=False)
        if not rating_df.empty:
            display(Markdown("### ⭐ Рейтинг"))
            display(rating_df)
    else:
        print("⚠️ Версия модели не найдена для отображения результатов.")
